# Lab 11: CNN Model for Time-Series Forecasting

**Name:** Zubair Moeen  
**Reg Number:** 22jzele0463  
**Lab:** Machine Learning Lab  
**Lab Supervisor:** Engr.Irshad Ullah  
**University:** UET Peshawar - Campus Nowshera 

## Lab Overview
This notebook develops a one-dimensional CNN model for time-series forecasting. The code defines a Conv1D architecture, sets up checkpoints and training callbacks, loads preprocessed time-series data, trains the model, and evaluates forecasting performance.

## Learning Objectives
- Import TensorFlow/Keras, Scikit-learn metrics, and custom time-series utilities.
- Define a CNN architecture using Conv1D layers for sequential data.
- Configure callbacks, optimizer, checkpoints, and training paths.
- Load train, validation, and test datasets for forecasting.
- Evaluate CNN forecasting performance using standard regression metrics.


## Section 1: Working Directory and Library Imports
This section sets the Lab 10/11 path and imports all libraries required for CNN-based time-series forecasting.


In [1]:
import os
os.chdir(r'Z:\University\8th Semester\ML Lab\Lab 10,11')

In [2]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [3]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [4]:
def CNN():
    input_data = Input(shape=(time_steps, num_features))
    x1 = Conv1D(16, 2, activation="relu")(input_data)
    x2 = Conv1D(16, 2, activation="relu")(x1)
    flatten = Flatten()(x2)
    output_data = Dense(1)(flatten)
    model = Model(input_data, output_data)
    return model

In [5]:
model1 = CNN()
model1.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 24, 21)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 23, 16)         │           688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 22, 16)         │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 352)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           353 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,569 (6.13 KB)

 Trainable params: 1,569 (6.13 KB)

 Non-trainable params: 0 (0.00 B)

## Section 2: Model Parameters and CNN Architecture
The following cells define input shape parameters and build the Conv1D-based CNN model for forecasting.


In [6]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [7]:
checkpoints = r'Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'Z:\University\8th Semester\ML Lab\Lab 10,11'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [8]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [9]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =CNN()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [10]:
import os
path_dataset =r'Z:\University\8th Semester\ML Lab\Lab 10,11'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\engin\.conda\envs\ml\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

## Section 3: Training Configuration and Callback Setup
This section prepares checkpoint saving, monitoring, optimizer configuration, and model compilation for training.


In [11]:
time_steps=24
num_features=21

In [12]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.005377054214477539 sec


In [13]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,verbose = verbose)

Epoch 1/10
15/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.4563 - mae: 0.4563 - mape: 307.3348
Epoch 1: val_loss improved from None to 0.09181, saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0001-loss0.09.h5



Epoch 1: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0001-loss0.09.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.2419 - mae: 0.2419 - mape: 125.5113 - val_loss: 0.0918 - val_mae: 0.0918 - val_mape: 35.9322
Epoch 2/10
16/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1096 - mae: 0.1096 - mape: 54.1291 
Epoch 2: val_loss improved from 0.09181 to 0.08406, saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0002-loss0.08.h5



Epoch 2: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0002-loss0.08.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.1018 - mae: 0.1018 - mape: 47.1383 - val_loss: 0.0841 - val_mae: 0.0841 - val_mape: 30.3170
Epoch 3/10
25/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0773 - mae: 0.0773 - mape: 40.3046
Epoch 3: val_loss improved from 0.08406 to 0.06680, saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0003-loss0.07.h5



Epoch 3: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0003-loss0.07.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0766 - mae: 0.0766 - mape: 40.4238 - val_loss: 0.0668 - val_mae: 0.0668 - val_mape: 23.7448
Epoch 4/10
17/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0689 - mae: 0.0689 - mape: 35.1141 
Epoch 4: val_loss improved from 0.06680 to 0.06592, saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0004-loss0.07.h5



Epoch 4: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0004-loss0.07.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0714 - mae: 0.0714 - mape: 34.2290 - val_loss: 0.0659 - val_mae: 0.0659 - val_mape: 22.8842
Epoch 5/10
17/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0671 - mae: 0.0671 - mape: 33.4049 
Epoch 5: val_loss did not improve from 0.06592
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0679 - mae: 0.0679 - mape: 33.6470 - val_loss: 0.0678 - val_mae: 0.0678 - val_mape: 23.5659
Epoch 6/10
16/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0665 - mae: 0.0665 - mape: 25.8032 
Epoch 6: val_loss improved from 0.06592 to 0.05920, saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0006-loss0.06.h5



Epoch 6: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0006-loss0.06.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0607 - mae: 0.0607 - mape: 29.9183 - val_loss: 0.0592 - val_mae: 0.0592 - val_mape: 20.8335
Epoch 7/10
15/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0486 - mae: 0.0486 - mape: 21.6046 
Epoch 7: val_loss did not improve from 0.05920
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0517 - mae: 0.0517 - mape: 26.1634 - val_loss: 0.0607 - val_mae: 0.0607 - val_mape: 19.6713
Epoch 8/10
15/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0513 - mae: 0.0513 - mape: 34.8631 
Epoch 8: val_loss did not improve from 0.05920
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0510 - mae: 0.0510 - mape: 24.0366 - val_loss: 0.0693 - val_mae: 0.0693 - val_mape: 23.9246
Epoch 9/10
15/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0466 - mae: 0.0466 - mape: 20.1145 
Epoch 9: val_loss did not improve from 0.05920
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - l


Epoch 10: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0010-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 0.0461 - mae: 0.0461 - mape: 21.9768 - val_loss: 0.0422 - val_mae: 0.0422 - val_mape: 13.9762


In [14]:

model = load_model(r'Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0010-loss0.04.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
Mean Absolute Error (MAE): 3012.93
Median Absolute Error (MedAE): 2797.2
Mean Squared Error (MSE): 9291765.88
Root Mean Squared Error (RMSE): 3048.24
Mean Absolute Percentage Error (MAPE): 19.2 %
Median Absolute Percentage Error (MDAPE): 18.07 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


In [15]:
checkpoints = r'Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0010-loss0.04.h5'
model=r'Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0010-loss0.04.h5'
start_epoch= 58

## Section 4: Dataset Loading, Prediction, and Evaluation
The remaining cells load the data, prepare it for the CNN model, generate predictions, and calculate forecasting metrics.


In [17]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
model_path = r'Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0010-loss0.04.h5'
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model_path))
    model = load_model(model_path, compile=False)

    opt = Adam(learning_rate=1e-4)
    model.compile(
        loss="mae",
        optimizer=opt,
        metrics=["mae", "mape"]
    )

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))

[INFO] loading Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0010-loss0.04.h5...
[INFO] old learning rate: 9.999999747378752e-05
[INFO] new learning rate: 9.999999747378752e-05


In [18]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
18/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0520 - mae: 0.0520 - mape: 23.6507
Epoch 1: val_loss improved from None to 0.05401, saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0010-loss0.04.h5



Epoch 1: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0010-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 0.0429 - mae: 0.0429 - mape: 20.2178 - val_loss: 0.0540 - val_mae: 0.0540 - val_mape: 18.8561
Epoch 2/10
17/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0374 - mae: 0.0374 - mape: 16.2471 
Epoch 2: val_loss did not improve from 0.05401
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0378 - mae: 0.0378 - mape: 18.0026 - val_loss: 0.0542 - val_mae: 0.0542 - val_mape: 19.0407
Epoch 3/10
15/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0392 - mae: 0.0392 - mape: 16.2542 
Epoch 3: val_loss did not improve from 0.05401
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0381 - mae: 0.0381 - mape: 18.3399 - val_loss: 0.0543 - val_mae: 0.0543 - val_mape: 18.9916
Epoch 4/10
14/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0391 - mae: 0.0391 - mape: 23.4542 
Epoch 4: val_loss did not improve from 0.05401
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - l


Epoch 10: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0010-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0354 - mae: 0.0354 - mape: 17.2777 - val_loss: 0.0521 - val_mae: 0.0521 - val_mape: 18.2817


In [19]:

model = load_model(r'Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0010-loss0.04.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
Mean Absolute Error (MAE): 2915.45
Median Absolute Error (MedAE): 2717.85
Mean Squared Error (MSE): 8717894.91
Root Mean Squared Error (RMSE): 2952.61
Mean Absolute Percentage Error (MAPE): 18.57 %
Median Absolute Percentage Error (MDAPE): 17.56 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## Final Conclusion
In this lab, a CNN model was applied to time-series forecasting. The notebook shows how Conv1D layers can extract local temporal patterns and how the trained model is evaluated using regression metrics.